In [ ]:
from pathlib import Path
import os

def _analysis_data_root():
    if os.environ.get("ANALYSIS_DATA_DIR"):
        return Path(os.environ["ANALYSIS_DATA_DIR"]).expanduser().resolve()
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / "analysis_data", base / "A_paper" / "analysis_data"):
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set ANALYSIS_DATA_DIR to the extracted analysis_data directory")

DATA_ROOT = _analysis_data_root()
RESULTS_ROOT = Path(os.environ.get("ANALYSIS_RESULTS_DIR", DATA_ROOT.parent / "analysis_code" / "results")).expanduser().resolve()


## Figure caption: Kruskal–Wallis test followed by BH-adjusted pairwise Mann–Whitney U tests versus Q4.

Full: Group differences were assessed using a Kruskal–Wallis test, followed by pairwise Mann–Whitney U tests comparing Q1, Q2, and Q3 with Q4, with Benjamini–Hochberg correction for multiple testing.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
B-cell ATAC accessibility metagene analysis

Analysis 1: split genes into Q1-Q4 by gene-attention score and compare ATAC accessibility.
Analysis 2: split the same genes into Q1-Q4 by B-cell RNA expression and compare ATAC accessibility.
Analysis 3: keep the gene-attention Q1-Q4 groups and compare the B-cell RNA expression of the corresponding genes.

All analyses use the full set of ATAC peaks only. The alternative peak set of
5,000 HVG peaks **is not analysed**: the paper's B-cell panel uses the all-peaks
grouping, so this notebook produces no HVG profile, no all-vs-HVG5000 comparison
figure, and no HVG peak-accessibility table.

X axis:
    -100 kb -> TSS -> scaled gene body -> TES -> +100 kb

Y axis:
    Mean B-cell peak accessibility

Inputs
------
1. gene_rank.csv
2. human ATAC h5ad
3. human RNA h5ad
4. NCBI RefSeq GRCh38 GTF

Outputs
-------
Gene annotation:
    Bcell_gene_rank_with_coordinates.csv
    Bcell_genes_unmatched_to_GTF.csv

ATAC peak accessibility:
    Bcell_all_peak_accessibility.csv

RNA expression:
    Bcell_ranked_gene_RNA_expression.csv
    Bcell_genes_with_RNA_expression_quantile.csv

Gene-attention grouping:
    Bcell_gene_attention_all_peaks_profile.pdf/png

RNA-expression grouping:
    Bcell_RNA_expression_all_peaks_profile.pdf/png

Gene-attention grouping -> RNA expression:
    Bcell_gene_attention_Q1Q4_RNA_expression.pdf/png
    Bcell_gene_attention_Q1Q4_RNA_expression_summary.csv
    Bcell_gene_attention_Q1Q4_RNA_expression_genes.csv
"""

from __future__ import annotations

import gc
import gzip
import re
from pathlib import Path
from typing import Dict, Optional

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats


# ============================================================
# 1. Paths
# ============================================================

BASE_DIR = DATA_ROOT / "Figure6/pbmc"

GENE_RANK_FILE = (
    BASE_DIR
    / "gene_rank.csv"
)

GTF_FILE = (
    BASE_DIR
    / "GRCh38.p14_genomic.gtf.gz"
)

ATAC_H5AD_FILE = BASE_DIR / "homo_sapiens_atac.h5ad"

RNA_H5AD_FILE = BASE_DIR / "homo_sapiens_rna.h5ad"

OUTPUT_DIR = (
    RESULTS_ROOT
    / "Figure6/pbmc/gene_peak_profiles"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Cell-type parameters
# ============================================================

ATAC_CELLTYPE_COLUMN = "Celltype"
RNA_CELLTYPE_COLUMN = "Celltype"

# Exact match, ignoring case and surrounding whitespace.
# The script prints the actual labels; if they differ, change them here.
BCELL_LABELS = [
    "B cell",
    "B cells",
    "B",
    "Naive B",
    "Memory B",
    "Intermediate B",
    "B naive",
    "B memory",
    "B intermediate",
]

ATAC_COUNTS_LAYER = "counts"
RNA_COUNTS_LAYER = "counts"


# ============================================================
# 3. ATAC accessibility parameters
# ============================================================

# Optional:
# mean_normalized
# fraction_accessible
# pseudobulk_cpm
# mean_raw_count
ACCESSIBILITY_METRIC = "mean_normalized"

# mean_normalized:
# Normalise each B cell's ATAC counts to 10,000,
# then compute the per-peak mean across B cells.
ATAC_TARGET_SUM = 10_000

# pseudobulk_cpm may be set to True;
# mean_normalized should stay False.
LOG1P_ACCESSIBILITY = False


# ============================================================
# 4. RNA expression parameters
# ============================================================

RNA_TARGET_SUM = 10_000

# Expression calculation:
# each B cell's RNA counts / cell total x 10,000
# then log1p, and finally average across B cells.
RNA_EXPRESSION_COLUMN = (
    "rna_mean_log_normalized_expression"
)


# ============================================================
# 5. Metagene parameters
# ============================================================

FLANK_SIZE = 100_000

UPSTREAM_BINS = 100
GENE_BODY_BINS = 100
DOWNSTREAM_BINS = 100

N_GENE_QUANTILES = 4

KEEP_STANDARD_CHROMOSOMES = True
EXCLUDE_MITOCHONDRIAL = True

SMOOTH_PROFILE = True
SMOOTH_WINDOW = 7

SHOW_SEM = True


# ============================================================
# 6. Plot settings
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["font.family"] = "Arial"


# ============================================================
# 7. Output paths
# ============================================================

GENE_COORDINATE_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_gene_rank_with_coordinates.csv"
)

UNMATCHED_GENE_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_genes_unmatched_to_GTF.csv"
)

ALL_PEAK_ACTIVITY_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_all_peak_accessibility.csv"
)

RNA_EXPRESSION_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_ranked_gene_RNA_expression.csv"
)

RNA_GENE_QUANTILE_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_genes_with_RNA_expression_quantile.csv"
)

# Gene-attention grouping results
ATTENTION_ALL_MATRIX_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_gene_attention_all_peaks_gene_bin_matrix.csv"
)

ATTENTION_ALL_SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_gene_attention_all_peaks_summary.csv"
)

ATTENTION_ALL_FIGURE_PREFIX = (
    OUTPUT_DIR
    / "Bcell_gene_attention_all_peaks_profile"
)


# RNA-expression grouping results
RNA_ALL_MATRIX_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_RNA_expression_all_peaks_gene_bin_matrix.csv"
)

RNA_ALL_SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_RNA_expression_all_peaks_summary.csv"
)

RNA_ALL_FIGURE_PREFIX = (
    OUTPUT_DIR
    / "Bcell_RNA_expression_all_peaks_profile"
)

# RNA expression for the gene-attention Q1-Q4 groups
ATTENTION_EXPRESSION_GENE_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_gene_attention_Q1Q4_RNA_expression_genes.csv"
)

ATTENTION_EXPRESSION_SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_gene_attention_Q1Q4_RNA_expression_summary.csv"
)

ATTENTION_EXPRESSION_FIGURE_PREFIX = (
    OUTPUT_DIR
    / "Bcell_gene_attention_Q1Q4_RNA_expression"
)

ATTENTION_EXPRESSION_STATS_OUTPUT = (
    OUTPUT_DIR
    / "Bcell_gene_attention_Q1Q4_RNA_expression_pairwise_stats.csv"
)


# ============================================================
# 8. Shared functions
# ============================================================

def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(
            f"File not found: {path}"
        )


def strip_ensembl_version(value: str) -> str:
    return re.sub(
        r"\.\d+$",
        "",
        str(value),
    )


def get_standard_chromosomes() -> set[str]:
    return {
        *(f"chr{i}" for i in range(1, 23)),
        "chrX",
        "chrY",
        "chrM",
    }


def normalize_chromosome(
    chromosome: str,
) -> Optional[str]:

    chromosome = str(chromosome).strip()

    if not chromosome:
        return None

    if chromosome.startswith("chr"):
        if chromosome == "chrMT":
            return "chrM"

        return chromosome

    if chromosome.startswith("NC_012920"):
        return "chrM"

    match = re.match(
        r"^NC_0*(\d+)(?:\.\d+)?$",
        chromosome,
    )

    if match:
        number = int(match.group(1))

        if 1 <= number <= 22:
            return f"chr{number}"

        if number == 23:
            return "chrX"

        if number == 24:
            return "chrY"

    chromosome_upper = chromosome.upper()

    if chromosome_upper in {
        str(i)
        for i in range(1, 23)
    }:
        return f"chr{chromosome_upper}"

    if chromosome_upper == "X":
        return "chrX"

    if chromosome_upper == "Y":
        return "chrY"

    if chromosome_upper in {"M", "MT"}:
        return "chrM"

    return chromosome


def parse_gtf_attributes(
    attribute_string: str,
) -> Dict[str, str]:

    attributes: Dict[str, str] = {}

    for item in (
        attribute_string
        .strip()
        .strip(";")
        .split(";")
    ):
        item = item.strip()

        if not item:
            continue

        match = re.match(
            r'^([^\s]+)\s+"(.*)"$',
            item,
        )

        if match:
            key, value = match.groups()
            attributes[key] = value
            continue

        parts = item.split(maxsplit=1)

        if len(parts) == 2:
            attributes[parts[0]] = (
                parts[1].strip('"')
            )

    return attributes


def rolling_smooth(
    values: np.ndarray,
    window: int,
) -> np.ndarray:

    if window <= 1:
        return values.copy()

    return (
        pd.Series(values)
        .rolling(
            window=window,
            center=True,
            min_periods=1,
        )
        .mean()
        .to_numpy()
    )


def select_bcell_mask(
    obs: pd.DataFrame,
    celltype_column: str,
    dataset_name: str,
) -> np.ndarray:

    if celltype_column not in obs.columns:
        raise KeyError(
            f"Column not found in {dataset_name}.obs: "
            f"{celltype_column}"
        )

    labels = (
        obs[celltype_column]
        .astype(str)
        .str.strip()
    )

    print(
        f"\n{dataset_name} cell-type labels:"
    )

    print(
        labels.value_counts().to_string()
    )

    target_labels = {
        str(label).strip().lower()
        for label in BCELL_LABELS
    }

    mask = (
        labels
        .str.lower()
        .isin(target_labels)
        .to_numpy()
    )

    n_selected = int(mask.sum())

    if n_selected == 0:
        raise ValueError(
            f"No B cells matched in {dataset_name}.\n"
            f"Current BCELL_LABELS={BCELL_LABELS}\n"
            "Update BCELL_LABELS according to the labels printed above."
        )

    print(
        f"{dataset_name} selected B cells: "
        f"{n_selected:,}/{len(obs):,}"
    )

    return mask


# ============================================================
# 9. Read the gene rank
# ============================================================

def load_gene_rank(
    path: Path,
) -> pd.DataFrame:

    require_file(path)

    df = pd.read_csv(path)

    required_columns = {
        "score",
        "gene_symbol",
        "gene_id",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"{path} is missing required columns: {sorted(missing)}\n"
            f"Current columns: {df.columns.tolist()}"
        )

    df = df.copy()

    df["gene_id"] = (
        df["gene_id"]
        .astype(str)
        .map(strip_ensembl_version)
    )

    df["gene_symbol"] = (
        df["gene_symbol"]
        .astype(str)
        .str.strip()
    )

    df["gene_symbol_upper"] = (
        df["gene_symbol"]
        .str.upper()
    )

    df["gene_attention"] = pd.to_numeric(
        df["score"],
        errors="coerce",
    )

    df = df.dropna(
        subset=[
            "gene_id",
            "gene_symbol",
            "gene_attention",
        ]
    )

    df = (
        df.sort_values(
            "gene_attention",
            ascending=False,
        )
        .drop_duplicates(
            subset=["gene_id"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    return df[
        [
            "gene_id",
            "gene_symbol",
            "gene_symbol_upper",
            "gene_attention",
        ]
    ]


# ============================================================
# 10. Read gene coordinates from the NCBI RefSeq GTF
# ============================================================

def load_gene_coordinates_from_gtf(
    path: Path,
) -> pd.DataFrame:

    require_file(path)

    opener = (
        gzip.open
        if path.suffix == ".gz"
        else open
    )

    records = []

    with opener(
        path,
        "rt",
        encoding="utf-8",
    ) as handle:

        for line in handle:
            if (
                not line
                or line.startswith("#")
            ):
                continue

            fields = (
                line
                .rstrip("\n")
                .split("\t")
            )

            if len(fields) != 9:
                continue

            if fields[2] != "gene":
                continue

            try:
                start_1based = int(fields[3])
                end_1based = int(fields[4])
            except ValueError:
                continue

            strand = fields[6]

            if strand not in {"+", "-"}:
                continue

            attributes = parse_gtf_attributes(
                fields[8]
            )

            refseq_gene_id = (
                attributes.get("gene_id")
            )

            gene_symbol = (
                attributes.get("gene")
                or attributes.get("gene_name")
                or refseq_gene_id
            )

            if gene_symbol is None:
                continue

            chromosome = normalize_chromosome(
                fields[0]
            )

            if chromosome is None:
                continue

            gene_symbol = (
                str(gene_symbol)
                .strip()
            )

            gene_start = start_1based - 1
            gene_end = end_1based

            if gene_end <= gene_start:
                continue

            if strand == "+":
                tss = gene_start
                tes = gene_end
            else:
                tss = gene_end
                tes = gene_start

            records.append(
                {
                    "refseq_gene_id": refseq_gene_id,
                    "gtf_gene_symbol": gene_symbol,
                    "gene_symbol_upper": (
                        gene_symbol.upper()
                    ),
                    "chrom": chromosome,
                    "gene_start": gene_start,
                    "gene_end": gene_end,
                    "strand": strand,
                    "tss": tss,
                    "tes": tes,
                    "gene_length": (
                        gene_end - gene_start
                    ),
                    "gene_biotype": (
                        attributes.get("gene_biotype")
                        or attributes.get("gene_type")
                        or ""
                    ),
                }
            )

    coordinates = pd.DataFrame(records)

    if coordinates.empty:
        raise ValueError(
            f"Could not extract gene coordinates from the GTF: {path}"
        )

    standard_chromosomes = (
        get_standard_chromosomes()
    )

    coordinates[
        "is_standard_chromosome"
    ] = coordinates["chrom"].isin(
        standard_chromosomes
    )

    # The same symbol may occur on patch/alternate contigs.
    # Prefer standard chromosomes, then the longer record.
    coordinates = (
        coordinates.sort_values(
            [
                "gene_symbol_upper",
                "is_standard_chromosome",
                "gene_length",
            ],
            ascending=[
                True,
                False,
                False,
            ],
        )
        .drop_duplicates(
            subset=["gene_symbol_upper"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    return coordinates


# ============================================================
# 11. Gene attention Q1-Q4
# ============================================================

def assign_quantiles(
    genes: pd.DataFrame,
    value_column: str,
    output_column: str,
) -> pd.DataFrame:

    result = genes.dropna(
        subset=[value_column]
    ).copy()

    if len(result) < N_GENE_QUANTILES:
        raise ValueError(
            f"{value_column} has only "
            f"{len(result)} valid genes and cannot be split into "
            f"{N_GENE_QUANTILES} groups."
        )

    rank_values = (
        result[value_column]
        .rank(
            method="first",
            ascending=True,
        )
    )

    result[output_column] = pd.qcut(
        rank_values,
        q=N_GENE_QUANTILES,
        labels=[
            f"Q{i}"
            for i in range(
                1,
                N_GENE_QUANTILES + 1,
            )
        ],
    ).astype(str)

    return result


# ============================================================
# 12. Parse peak coordinates
# ============================================================

def parse_peak_intervals(
    intervals: pd.Series,
) -> pd.DataFrame:

    interval_strings = (
        intervals
        .astype(str)
        .str.strip()
    )

    # Preferred: chr1:100-200
    coordinates = interval_strings.str.extract(
        r"^(.+?):(\d+)-(\d+)$"
    )

    coordinates.columns = [
        "chrom",
        "peak_start",
        "peak_end",
    ]

    unresolved = (
        coordinates.isna().any(axis=1)
    )

    # Also accepts chr1_100_200
    if unresolved.any():
        fallback = (
            interval_strings.loc[unresolved]
            .str.extract(
                r"^(.+?)_(\d+)_(\d+)$"
            )
        )

        fallback.columns = [
            "chrom",
            "peak_start",
            "peak_end",
        ]

        coordinates.loc[
            unresolved,
            fallback.columns,
        ] = fallback.values

    coordinates["chrom"] = (
        coordinates["chrom"]
        .map(normalize_chromosome)
    )

    coordinates["peak_start"] = pd.to_numeric(
        coordinates["peak_start"],
        errors="coerce",
    )

    coordinates["peak_end"] = pd.to_numeric(
        coordinates["peak_end"],
        errors="coerce",
    )

    return coordinates


# ============================================================
# 13. Compute B-cell peak accessibility
# ============================================================

def calculate_bcell_peak_accessibility(
    adata_atac: ad.AnnData,
) -> pd.DataFrame:

    if ATAC_COUNTS_LAYER not in adata_atac.layers:
        raise KeyError(
            f"Not present in ATAC layers: "
            f"{ATAC_COUNTS_LAYER}"
        )

    bcell_mask = select_bcell_mask(
        obs=adata_atac.obs,
        celltype_column=(
            ATAC_CELLTYPE_COLUMN
        ),
        dataset_name="ATAC",
    )

    counts = adata_atac.layers[
        ATAC_COUNTS_LAYER
    ][
        bcell_mask,
        :
    ]

    n_bcells = int(bcell_mask.sum())

    if sp.issparse(counts):
        counts = (
            counts
            .tocsr()
            .astype(np.float64)
        )
    else:
        counts = np.asarray(
            counts,
            dtype=np.float64,
        )

    # --------------------------------------------------------
    # Raw statistics
    # --------------------------------------------------------

    if sp.issparse(counts):
        nonzero_cell_count = np.asarray(
            (counts > 0).sum(axis=0)
        ).ravel()

        pseudobulk_count = np.asarray(
            counts.sum(axis=0)
        ).ravel()

        mean_raw_count = np.asarray(
            counts.mean(axis=0)
        ).ravel()

        cell_totals = np.asarray(
            counts.sum(axis=1)
        ).ravel()

    else:
        nonzero_cell_count = (
            counts > 0
        ).sum(axis=0)

        pseudobulk_count = counts.sum(
            axis=0
        )

        mean_raw_count = counts.mean(
            axis=0
        )

        cell_totals = counts.sum(
            axis=1
        )

    fraction_accessible = (
        nonzero_cell_count
        / n_bcells
    )

    total_fragments = float(
        pseudobulk_count.sum()
    )

    if total_fragments <= 0:
        raise ValueError(
            "The total B-cell ATAC count is 0."
        )

    pseudobulk_cpm = (
        pseudobulk_count
        / total_fragments
        * 1e6
    )

    # --------------------------------------------------------
    # Per-cell library-size normalisation
    # No full normalised matrix is created
    # --------------------------------------------------------

    scale_factors = np.divide(
        ATAC_TARGET_SUM,
        cell_totals,
        out=np.zeros_like(
            cell_totals,
            dtype=float,
        ),
        where=cell_totals > 0,
    )

    if sp.issparse(counts):
        mean_normalized = np.asarray(
            counts.T.dot(scale_factors)
        ).ravel() / n_bcells

    else:
        mean_normalized = (
            counts.T @ scale_factors
        ) / n_bcells

    # --------------------------------------------------------
    # Peak IDs
    # --------------------------------------------------------

    if "interval" in adata_atac.var.columns:
        peak_ids = (
            adata_atac.var["interval"]
            .astype(str)
            .to_numpy()
        )
    else:
        peak_ids = (
            adata_atac.var_names
            .astype(str)
            .to_numpy()
        )

    coordinates = parse_peak_intervals(
        pd.Series(peak_ids)
    )

    invalid = (
        coordinates.isna().any(axis=1)
    )

    if invalid.any():
        print(
            "Warning: ATAC peaks whose coordinates could not be parsed: "
            f"{invalid.sum():,}"
        )

        print(
            pd.Series(peak_ids)
            .loc[invalid]
            .head(10)
            .tolist()
        )

    result = pd.DataFrame(
        {
            "peak_id": peak_ids,
            "chrom": (
                coordinates["chrom"]
                .to_numpy()
            ),
            "peak_start": (
                coordinates["peak_start"]
                .to_numpy()
            ),
            "peak_end": (
                coordinates["peak_end"]
                .to_numpy()
            ),
            "nonzero_cell_count": (
                nonzero_cell_count
            ),
            "fraction_accessible": (
                fraction_accessible
            ),
            "mean_raw_count": (
                mean_raw_count
            ),
            "pseudobulk_count": (
                pseudobulk_count
            ),
            "pseudobulk_cpm": (
                pseudobulk_cpm
            ),
            "mean_normalized": (
                mean_normalized
            ),
        }
    )

    # Save the HVG-related information in ATAC.var
    for column in [
        "highly_variable",
        "means",
        "dispersions",
        "dispersions_norm",
        "feature_types",
        "genome",
        "embed_statue",
    ]:
        if column in adata_atac.var.columns:
            result[column] = (
                adata_atac.var[column]
                .to_numpy()
            )

    result = result.dropna(
        subset=[
            "chrom",
            "peak_start",
            "peak_end",
        ]
    ).copy()

    result["peak_start"] = (
        result["peak_start"]
        .astype(int)
    )

    result["peak_end"] = (
        result["peak_end"]
        .astype(int)
    )

    result = result[
        result["peak_end"]
        > result["peak_start"]
    ].copy()

    allowed_metrics = {
        "mean_normalized",
        "fraction_accessible",
        "pseudobulk_cpm",
        "mean_raw_count",
    }

    if (
        ACCESSIBILITY_METRIC
        not in allowed_metrics
    ):
        raise ValueError(
            "ACCESSIBILITY_METRIC must be one of: "
            f"{sorted(allowed_metrics)}"
        )

    result["peak_accessibility"] = (
        pd.to_numeric(
            result[ACCESSIBILITY_METRIC],
            errors="coerce",
        )
    )

    if LOG1P_ACCESSIBILITY:
        result["peak_accessibility"] = (
            np.log1p(
                result["peak_accessibility"]
            )
        )

    result["peak_length"] = (
        result["peak_end"]
        - result["peak_start"]
    )

    return result.reset_index(drop=True)


# ============================================================
# 14. RNA feature lookup
# ============================================================

def add_lookup_value(
    lookup: Dict[str, list[int]],
    key: str,
    index: int,
) -> None:

    key = str(key).strip()

    if (
        not key
        or key.lower() in {
            "nan",
            "none",
        }
    ):
        return

    lookup.setdefault(
        key,
        [],
    ).append(index)


def build_rna_feature_lookup(
    adata_rna: ad.AnnData,
) -> tuple[
    Dict[str, list[int]],
    Dict[str, list[int]],
]:

    gene_id_lookup: Dict[
        str,
        list[int]
    ] = {}

    gene_symbol_lookup: Dict[
        str,
        list[int]
    ] = {}

    # Try var_names as both ID and symbol
    for index, value in enumerate(
        adata_rna.var_names.astype(str)
    ):
        add_lookup_value(
            gene_id_lookup,
            strip_ensembl_version(value),
            index,
        )

        add_lookup_value(
            gene_symbol_lookup,
            value.upper(),
            index,
        )

    gene_id_columns = [
        "gene_id",
        "gene_ids",
        "ensembl_id",
        "ensembl_ids",
    ]

    gene_symbol_columns = [
        "gene_symbol",
        "gene_symbols",
        "gene_name",
        "gene_names",
        "feature_name",
        "features",
    ]

    for column in gene_id_columns:
        if column not in adata_rna.var.columns:
            continue

        values = (
            adata_rna.var[column]
            .astype(str)
            .to_numpy()
        )

        for index, value in enumerate(values):
            add_lookup_value(
                gene_id_lookup,
                strip_ensembl_version(value),
                index,
            )

    for column in gene_symbol_columns:
        if column not in adata_rna.var.columns:
            continue

        values = (
            adata_rna.var[column]
            .astype(str)
            .to_numpy()
        )

        for index, value in enumerate(values):
            add_lookup_value(
                gene_symbol_lookup,
                value.strip().upper(),
                index,
            )

    gene_id_lookup = {
        key: sorted(set(indices))
        for key, indices
        in gene_id_lookup.items()
    }

    gene_symbol_lookup = {
        key: sorted(set(indices))
        for key, indices
        in gene_symbol_lookup.items()
    }

    return (
        gene_id_lookup,
        gene_symbol_lookup,
    )


# ============================================================
# 15. Compute B-cell RNA expression
# ============================================================

def calculate_bcell_gene_expression(
    adata_rna: ad.AnnData,
    genes: pd.DataFrame,
) -> pd.DataFrame:

    bcell_mask = select_bcell_mask(
        obs=adata_rna.obs,
        celltype_column=(
            RNA_CELLTYPE_COLUMN
        ),
        dataset_name="RNA",
    )

    if RNA_COUNTS_LAYER in adata_rna.layers:
        matrix = adata_rna.layers[
            RNA_COUNTS_LAYER
        ][
            bcell_mask,
            :
        ]

        print(
            f"RNA matrix source: "
            f"layers['{RNA_COUNTS_LAYER}']"
        )

    else:
        matrix = adata_rna.X[
            bcell_mask,
            :
        ]

        print(
            "Warning: RNA has no counts layer; "
            "using adata.X."
        )

    if sp.issparse(matrix):
        matrix = (
            matrix
            .tocsr()
            .astype(np.float64)
        )

        cell_totals = np.asarray(
            matrix.sum(axis=1)
        ).ravel()

    else:
        matrix = np.asarray(
            matrix,
            dtype=np.float64,
        )

        cell_totals = matrix.sum(
            axis=1
        )

    scale_factors = np.divide(
        RNA_TARGET_SUM,
        cell_totals,
        out=np.zeros_like(
            cell_totals,
            dtype=float,
        ),
        where=cell_totals > 0,
    )

    (
        gene_id_lookup,
        gene_symbol_lookup,
    ) = build_rna_feature_lookup(
        adata_rna
    )

    records = []

    for gene in genes.itertuples(
        index=False
    ):
        gene_id = strip_ensembl_version(
            gene.gene_id
        )

        gene_symbol = str(
            gene.gene_symbol
        ).strip()

        gene_symbol_upper = (
            gene_symbol.upper()
        )

        matched_indices = []
        match_method = ""

        if gene_id in gene_id_lookup:
            matched_indices = (
                gene_id_lookup[gene_id]
            )

            match_method = "gene_id"

        elif (
            gene_symbol_upper
            in gene_symbol_lookup
        ):
            matched_indices = (
                gene_symbol_lookup[
                    gene_symbol_upper
                ]
            )

            match_method = "gene_symbol"

        if not matched_indices:
            records.append(
                {
                    "gene_id": gene_id,
                    "gene_symbol": gene_symbol,
                    "rna_match_status": (
                        "unmatched"
                    ),
                    "rna_match_method": "",
                    "rna_n_features": 0,
                    RNA_EXPRESSION_COLUMN: (
                        np.nan
                    ),
                    "rna_fraction_expressing": (
                        np.nan
                    ),
                    "rna_pseudobulk_count": (
                        np.nan
                    ),
                }
            )

            continue

        selected = matrix[
            :,
            matched_indices,
        ]

        if sp.issparse(selected):
            raw_per_cell = np.asarray(
                selected.sum(axis=1)
            ).ravel()
        else:
            raw_per_cell = np.asarray(
                selected.sum(axis=1)
            ).ravel()

        normalized_per_cell = (
            raw_per_cell
            * scale_factors
        )

        log_normalized_per_cell = (
            np.log1p(
                normalized_per_cell
            )
        )

        records.append(
            {
                "gene_id": gene_id,
                "gene_symbol": gene_symbol,
                "rna_match_status": (
                    "matched"
                ),
                "rna_match_method": (
                    match_method
                ),
                "rna_n_features": (
                    len(matched_indices)
                ),
                RNA_EXPRESSION_COLUMN: (
                    float(
                        np.mean(
                            log_normalized_per_cell
                        )
                    )
                ),
                "rna_fraction_expressing": (
                    float(
                        np.mean(
                            raw_per_cell > 0
                        )
                    )
                ),
                "rna_pseudobulk_count": (
                    float(
                        np.sum(raw_per_cell)
                    )
                ),
            }
        )

    return pd.DataFrame(records)


# ============================================================
# 16. Build the metagene bins
# ============================================================

def build_bins_for_gene(
    gene_start: int,
    gene_end: int,
    strand: str,
) -> pd.DataFrame:

    if gene_end <= gene_start:
        raise ValueError(
            f"Invalid gene interval: "
            f"{gene_start}-{gene_end}"
        )

    if strand not in {"+", "-"}:
        raise ValueError(
            f"Invalid strand: {strand}"
        )

    tss = (
        gene_start
        if strand == "+"
        else gene_end
    )

    tes = (
        gene_end
        if strand == "+"
        else gene_start
    )

    records = []

    # --------------------------------------------------------
    # Upstream: -100 kb to TSS
    # --------------------------------------------------------

    upstream_edges = np.linspace(
        0,
        FLANK_SIZE,
        UPSTREAM_BINS + 1,
    )

    for index in range(
        UPSTREAM_BINS
    ):
        relative_left = int(
            round(upstream_edges[index])
        )

        relative_right = int(
            round(
                upstream_edges[
                    index + 1
                ]
            )
        )

        if strand == "+":
            genomic_start = (
                tss - relative_right
            )

            genomic_end = (
                tss - relative_left
            )

        else:
            genomic_start = (
                tss + relative_left
            )

            genomic_end = (
                tss + relative_right
            )

        records.append(
            {
                "bin_index": index,
                "region": "upstream",
                "genomic_start": (
                    genomic_start
                ),
                "genomic_end": (
                    genomic_end
                ),
            }
        )

    # --------------------------------------------------------
    # Gene body: scaled to a fixed number of bins along the transcription direction
    # --------------------------------------------------------

    body_edges = np.linspace(
        gene_start,
        gene_end,
        GENE_BODY_BINS + 1,
    )

    for body_index in range(
        GENE_BODY_BINS
    ):
        if strand == "+":
            genomic_index = body_index

        else:
            genomic_index = (
                GENE_BODY_BINS
                - body_index
                - 1
            )

        genomic_start = int(
            round(
                body_edges[
                    genomic_index
                ]
            )
        )

        genomic_end = int(
            round(
                body_edges[
                    genomic_index + 1
                ]
            )
        )

        if genomic_end <= genomic_start:
            genomic_end = (
                genomic_start + 1
            )

        records.append(
            {
                "bin_index": (
                    UPSTREAM_BINS
                    + body_index
                ),
                "region": "gene_body",
                "genomic_start": (
                    genomic_start
                ),
                "genomic_end": (
                    genomic_end
                ),
            }
        )

    # --------------------------------------------------------
    # Downstream: TES to +100 kb
    # --------------------------------------------------------

    downstream_edges = np.linspace(
        0,
        FLANK_SIZE,
        DOWNSTREAM_BINS + 1,
    )

    for index in range(
        DOWNSTREAM_BINS
    ):
        relative_left = int(
            round(
                downstream_edges[index]
            )
        )

        relative_right = int(
            round(
                downstream_edges[
                    index + 1
                ]
            )
        )

        if strand == "+":
            genomic_start = (
                tes + relative_left
            )

            genomic_end = (
                tes + relative_right
            )

        else:
            genomic_start = (
                tes - relative_right
            )

            genomic_end = (
                tes - relative_left
            )

        records.append(
            {
                "bin_index": (
                    UPSTREAM_BINS
                    + GENE_BODY_BINS
                    + index
                ),
                "region": "downstream",
                "genomic_start": (
                    genomic_start
                ),
                "genomic_end": (
                    genomic_end
                ),
            }
        )

    bins = pd.DataFrame(records)

    bins["genomic_start"] = (
        bins["genomic_start"]
        .clip(lower=0)
    )

    bins["bin_length"] = (
        bins["genomic_end"]
        - bins["genomic_start"]
    )

    bins = bins[
        bins["bin_length"] > 0
    ].copy()

    bins["normalized_position"] = (
        bins["bin_index"] + 0.5
    )

    return bins


# ============================================================
# 17. Compute the single-gene metagene profile
# ============================================================

def calculate_gene_profile(
    gene: pd.Series,
    chromosome_peaks: pd.DataFrame,
    peak_set_name: str,
    group_column: str,
) -> pd.DataFrame:

    bins = build_bins_for_gene(
        gene_start=int(
            gene["gene_start"]
        ),
        gene_end=int(
            gene["gene_end"]
        ),
        strand=str(
            gene["strand"]
        ),
    )

    region_start = int(
        bins["genomic_start"].min()
    )

    region_end = int(
        bins["genomic_end"].max()
    )

    candidate_peaks = chromosome_peaks[
        (
            chromosome_peaks[
                "peak_end"
            ]
            > region_start
        )
        &
        (
            chromosome_peaks[
                "peak_start"
            ]
            < region_end
        )
    ]

    signals = np.zeros(
        len(bins),
        dtype=float,
    )

    peak_counts = np.zeros(
        len(bins),
        dtype=int,
    )

    if not candidate_peaks.empty:
        peak_starts = (
            candidate_peaks[
                "peak_start"
            ]
            .to_numpy(dtype=int)
        )

        peak_ends = (
            candidate_peaks[
                "peak_end"
            ]
            .to_numpy(dtype=int)
        )

        peak_values = (
            candidate_peaks[
                "peak_accessibility"
            ]
            .to_numpy(dtype=float)
        )

        for local_index, bin_row in enumerate(
            bins.itertuples(index=False)
        ):
            overlap_start = np.maximum(
                peak_starts,
                int(bin_row.genomic_start),
            )

            overlap_end = np.minimum(
                peak_ends,
                int(bin_row.genomic_end),
            )

            overlap_bp = np.maximum(
                0,
                overlap_end - overlap_start,
            )

            overlapping = (
                overlap_bp > 0
            )

            if not overlapping.any():
                continue

            signals[local_index] = (
                np.sum(
                    peak_values[overlapping]
                    * overlap_bp[overlapping]
                )
                / float(bin_row.bin_length)
            )

            peak_counts[local_index] = (
                int(overlapping.sum())
            )

    bins[
        "peak_accessibility_signal"
    ] = signals

    bins[
        "overlapping_peak_count"
    ] = peak_counts

    bins["peak_set"] = peak_set_name
    bins["gene_id"] = gene["gene_id"]
    bins["gene_symbol"] = gene["gene_symbol"]
    bins["chrom"] = gene["chrom"]
    bins["strand"] = gene["strand"]
    bins[group_column] = gene[group_column]

    if "gene_attention" in gene.index:
        bins["gene_attention"] = (
            gene["gene_attention"]
        )

    if (
        RNA_EXPRESSION_COLUMN
        in gene.index
    ):
        bins[
            RNA_EXPRESSION_COLUMN
        ] = gene[
            RNA_EXPRESSION_COLUMN
        ]

    return bins


# ============================================================
# 18. Profiles for all genes
# ============================================================

def calculate_all_profiles(
    genes: pd.DataFrame,
    peaks: pd.DataFrame,
    peak_set_name: str,
    group_column: str,
) -> pd.DataFrame:

    peaks_by_chromosome = {
        chromosome: (
            current
            .sort_values("peak_start")
            .reset_index(drop=True)
        )
        for chromosome, current
        in peaks.groupby(
            "chrom",
            sort=False,
        )
    }

    empty_peak_table = pd.DataFrame(
        columns=peaks.columns
    )

    profiles = []

    total_genes = len(genes)

    for processed_index, (_, gene) in enumerate(
        genes.iterrows(),
        start=1,
    ):
        chromosome_peaks = (
            peaks_by_chromosome.get(
                gene["chrom"],
                empty_peak_table,
            )
        )

        profiles.append(
            calculate_gene_profile(
                gene=gene,
                chromosome_peaks=(
                    chromosome_peaks
                ),
                peak_set_name=(
                    peak_set_name
                ),
                group_column=(
                    group_column
                ),
            )
        )

        if (
            processed_index == 1
            or processed_index % 20 == 0
            or processed_index
            == total_genes
        ):
            print(
                f"[{peak_set_name}; "
                f"{group_column}] "
                f"Processed genes: "
                f"{processed_index:,}/"
                f"{total_genes:,}"
            )

    if not profiles:
        raise ValueError(
            "No metagene profiles generated."
        )

    return pd.concat(
        profiles,
        ignore_index=True,
    )


# ============================================================
# 19. Summarise the Q1-Q4 profiles
# ============================================================

def summarize_profiles(
    profile_df: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:

    summary = (
        profile_df
        .groupby(
            [
                "peak_set",
                group_column,
                "bin_index",
                "normalized_position",
                "region",
            ],
            observed=True,
        )
        .agg(
            mean_peak_accessibility=(
                "peak_accessibility_signal",
                "mean",
            ),
            median_peak_accessibility=(
                "peak_accessibility_signal",
                "median",
            ),
            standard_deviation=(
                "peak_accessibility_signal",
                "std",
            ),
            n_genes=(
                "gene_id",
                "nunique",
            ),
            mean_overlapping_peak_count=(
                "overlapping_peak_count",
                "mean",
            ),
        )
        .reset_index()
    )

    summary[
        "standard_deviation"
    ] = (
        summary[
            "standard_deviation"
        ]
        .fillna(0)
    )

    summary["standard_error"] = (
        summary["standard_deviation"]
        / np.sqrt(
            summary["n_genes"]
        )
    )

    return summary


# ============================================================
# 20. Plotting functions
# ============================================================

def set_metagene_axis(
    ax: plt.Axes,
) -> None:

    tss_position = UPSTREAM_BINS

    tes_position = (
        UPSTREAM_BINS
        + GENE_BODY_BINS
    )

    total_bins = (
        UPSTREAM_BINS
        + GENE_BODY_BINS
        + DOWNSTREAM_BINS
    )

    ax.axvline(
        tss_position,
        linewidth=0.8,
        linestyle="--",
        alpha=0.7,
    )

    ax.axvline(
        tes_position,
        linewidth=0.8,
        linestyle="--",
        alpha=0.7,
    )

    ax.set_xlim(
        0,
        total_bins,
    )

    if FLANK_SIZE >= 1000:
        flank_label = (
            f"{FLANK_SIZE / 1000:g} kb"
        )
    else:
        flank_label = (
            f"{FLANK_SIZE} bp"
        )

    ax.set_xticks(
        [
            0,
            tss_position,
            tes_position,
            total_bins,
        ]
    )

    ax.set_xticklabels(
        [
            f"−{flank_label}",
            "TSS",
            "TES",
            f"+{flank_label}",
        ]
    )

    ax.set_xlabel(
        "Relative position across gene region"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def draw_profile_on_axis(
    ax: plt.Axes,
    summary: pd.DataFrame,
    group_column: str,
    title: str,
    legend_title: str,
) -> None:

    quantile_order = [
        f"Q{i}"
        for i in range(
            1,
            N_GENE_QUANTILES + 1,
        )
    ]

    for quantile in quantile_order:
        current = (
            summary[
                summary[group_column]
                == quantile
            ]
            .sort_values("bin_index")
        )

        if current.empty:
            continue

        x = current[
            "normalized_position"
        ].to_numpy(dtype=float)

        y = current[
            "mean_peak_accessibility"
        ].to_numpy(dtype=float)

        sem = current[
            "standard_error"
        ].to_numpy(dtype=float)

        if SMOOTH_PROFILE:
            y_plot = rolling_smooth(
                y,
                SMOOTH_WINDOW,
            )

            sem_plot = rolling_smooth(
                sem,
                SMOOTH_WINDOW,
            )

        else:
            y_plot = y
            sem_plot = sem

        n_genes = int(
            current["n_genes"].max()
        )

        ax.plot(
            x,
            y_plot,
            linewidth=2,
            label=(
                f"{quantile} "
                f"(n={n_genes})"
            ),
        )

        if SHOW_SEM:
            ax.fill_between(
                x,
                np.maximum(
                    0,
                    y_plot - sem_plot,
                ),
                y_plot + sem_plot,
                alpha=0.12,
                linewidth=0,
            )

    set_metagene_axis(ax)

    ax.set_ylabel(
        "Mean B-cell peak accessibility"
    )

    ax.set_title(title)

    ax.legend(
        title=legend_title,
        frameon=False,
    )


def save_single_profile(
    summary: pd.DataFrame,
    group_column: str,
    title: str,
    legend_title: str,
    output_prefix: Path,
) -> None:

    fig, ax = plt.subplots(
        figsize=(8.5, 5.5)
    )

    draw_profile_on_axis(
        ax=ax,
        summary=summary,
        group_column=group_column,
        title=title,
        legend_title=legend_title,
    )

    plt.tight_layout()

    pdf_file = output_prefix.with_suffix(
        ".pdf"
    )

    png_file = output_prefix.with_suffix(
        ".png"
    )

    plt.savefig(
        pdf_file,
        bbox_inches="tight",
    )

    plt.savefig(
        png_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    print(f"Saved: {pdf_file}")
    print(f"Saved: {png_file}")


# ============================================================
# 21. RNA expression for the gene-attention Q1-Q4 groups
# ============================================================

def summarize_expression_by_attention_quantile(
    genes_rna: pd.DataFrame,
) -> pd.DataFrame:
    """
    Summarise B-cell RNA expression by gene-attention Q1-Q4.

    Each row is one gene; the expression column is:
        rna_mean_log_normalized_expression
    """

    required_columns = {
        "gene_id",
        "gene_symbol",
        "gene_attention",
        "gene_attention_quantile",
        RNA_EXPRESSION_COLUMN,
    }

    missing = required_columns - set(
        genes_rna.columns
    )

    if missing:
        raise ValueError(
            "genes_rna is missing required columns: "
            f"{sorted(missing)}"
        )

    data = genes_rna.dropna(
        subset=[
            "gene_attention_quantile",
            RNA_EXPRESSION_COLUMN,
        ]
    ).copy()

    quantile_order = [
        f"Q{i}"
        for i in range(
            1,
            N_GENE_QUANTILES + 1,
        )
    ]

    data[
        "gene_attention_quantile"
    ] = pd.Categorical(
        data[
            "gene_attention_quantile"
        ],
        categories=quantile_order,
        ordered=True,
    )

    summary = (
        data
        .groupby(
            "gene_attention_quantile",
            observed=True,
        )
        .agg(
            n_genes=(
                "gene_id",
                "nunique",
            ),
            mean_expression=(
                RNA_EXPRESSION_COLUMN,
                "mean",
            ),
            median_expression=(
                RNA_EXPRESSION_COLUMN,
                "median",
            ),
            standard_deviation=(
                RNA_EXPRESSION_COLUMN,
                "std",
            ),
            expression_q25=(
                RNA_EXPRESSION_COLUMN,
                lambda values: values.quantile(0.25),
            ),
            expression_q75=(
                RNA_EXPRESSION_COLUMN,
                lambda values: values.quantile(0.75),
            ),
            mean_gene_attention=(
                "gene_attention",
                "mean",
            ),
            median_gene_attention=(
                "gene_attention",
                "median",
            ),
        )
        .reset_index()
    )

    summary[
        "standard_deviation"
    ] = summary[
        "standard_deviation"
    ].fillna(0)

    summary[
        "standard_error"
    ] = (
        summary[
            "standard_deviation"
        ]
        / np.sqrt(
            summary["n_genes"]
        )
    )

    summary[
        "gene_attention_quantile"
    ] = pd.Categorical(
        summary[
            "gene_attention_quantile"
        ],
        categories=quantile_order,
        ordered=True,
    )

    summary = (
        summary
        .sort_values(
            "gene_attention_quantile"
        )
        .reset_index(drop=True)
    )

    return summary


def pvalue_to_stars(p_value: float) -> str:
    if pd.isna(p_value):
        return "ns"
    if p_value < 1e-4:
        return "****"
    if p_value < 1e-3:
        return "***"
    if p_value < 1e-2:
        return "**"
    if p_value < 5e-2:
        return "*"
    return "ns"


def benjamini_hochberg(p_values: list[float]) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = np.empty(n, dtype=float)
    cumulative_min = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        value = ranked[i] * n / rank
        cumulative_min = min(cumulative_min, value)
        adjusted[i] = cumulative_min
    result = np.empty(n, dtype=float)
    result[order] = np.clip(adjusted, 0, 1)
    return result


def compute_attention_expression_pairwise_tests(
    genes_rna: pd.DataFrame,
) -> pd.DataFrame:
    quantile_order = [
        f"Q{i}"
        for i in range(
            1,
            N_GENE_QUANTILES + 1,
        )
    ]

    data = genes_rna.dropna(
        subset=[
            "gene_attention_quantile",
            RNA_EXPRESSION_COLUMN,
        ]
    ).copy()

    groups = {
        quantile: data.loc[
            data["gene_attention_quantile"] == quantile,
            RNA_EXPRESSION_COLUMN,
        ].to_numpy(dtype=float)
        for quantile in quantile_order
    }

    comparisons = [
        ("Q1", "Q4"),
        ("Q2", "Q4"),
        ("Q3", "Q4"),
    ]

    records = []
    raw_p_values = []

    for group1, group2 in comparisons:
        values1 = groups.get(group1, np.array([], dtype=float))
        values2 = groups.get(group2, np.array([], dtype=float))

        if len(values1) == 0 or len(values2) == 0:
            statistic = np.nan
            p_value = np.nan
        else:
            statistic, p_value = stats.mannwhitneyu(
                values1,
                values2,
                alternative="two-sided",
            )

        records.append(
            {
                "group1": group1,
                "group2": group2,
                "n_group1": len(values1),
                "n_group2": len(values2),
                "mean_group1": float(np.mean(values1)) if len(values1) > 0 else np.nan,
                "mean_group2": float(np.mean(values2)) if len(values2) > 0 else np.nan,
                "median_group1": float(np.median(values1)) if len(values1) > 0 else np.nan,
                "median_group2": float(np.median(values2)) if len(values2) > 0 else np.nan,
                "mannwhitney_u": statistic,
                "p_value": p_value,
            }
        )

        raw_p_values.append(p_value if not pd.isna(p_value) else 1.0)

    result = pd.DataFrame(records)

    if len(result) > 0:
        result["p_adj_bh"] = benjamini_hochberg(raw_p_values)
        result["significance"] = result["p_adj_bh"].map(pvalue_to_stars)

    return result


def add_significance_annotation(
    ax: plt.Axes,
    x1: float,
    x2: float,
    y: float,
    h: float,
    text: str,
) -> None:
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        linewidth=1.0,
    )
    ax.text(
        (x1 + x2) / 2,
        y + h,
        text,
        ha="center",
        va="bottom",
        fontsize=10,
    )


def plot_expression_by_attention_quantile(
    genes_rna: pd.DataFrame,
    summary: pd.DataFrame,
    output_prefix: Path,
    stats_output: Optional[Path] = None,
) -> None:
    """
    Plot B-cell RNA expression for the gene-attention Q1-Q4 groups.

    Layers:
    1. one jittered scatter point per gene;
    2. a box plot;
    3. significance stars for the Mann-Whitney U test against Q4 only (BH-corrected).
    """

    quantile_order = [
        f"Q{i}"
        for i in range(
            1,
            N_GENE_QUANTILES + 1,
        )
    ]

    data = genes_rna.dropna(
        subset=[
            "gene_attention_quantile",
            RNA_EXPRESSION_COLUMN,
        ]
    ).copy()

    data["gene_attention_quantile"] = pd.Categorical(
        data["gene_attention_quantile"],
        categories=quantile_order,
        ordered=True,
    )

    data = data.sort_values("gene_attention_quantile")

    expression_arrays = [
        data.loc[
            data["gene_attention_quantile"] == quantile,
            RNA_EXPRESSION_COLUMN,
        ].to_numpy(dtype=float)
        for quantile in quantile_order
    ]

    pairwise_stats = compute_attention_expression_pairwise_tests(
        genes_rna=data
    )

    if stats_output is not None:
        pairwise_stats.to_csv(
            stats_output,
            index=False,
        )
        print(f"Saved: {stats_output}")

    fig, ax = plt.subplots(figsize=(7.2, 5.8))
    positions = np.arange(1, len(quantile_order) + 1)

    ax.boxplot(
        expression_arrays,
        positions=positions,
        widths=0.55,
        patch_artist=False,
        showfliers=False,
        medianprops={"linewidth": 1.5},
        boxprops={"linewidth": 1.2},
        whiskerprops={"linewidth": 1.0},
        capprops={"linewidth": 1.0},
    )

    random_generator = np.random.default_rng(42)

    for position, values in zip(positions, expression_arrays):
        if len(values) == 0:
            continue

        jitter = random_generator.uniform(
            low=-0.18,
            high=0.18,
            size=len(values),
        )

        ax.scatter(
            np.full(len(values), position, dtype=float) + jitter,
            values,
            s=18,
            alpha=0.55,
            linewidths=0,
        )

    summary_indexed = (
        summary
        .set_index("gene_attention_quantile")
        .reindex(quantile_order)
    )

    group_labels = []
    for quantile in quantile_order:
        if quantile in summary_indexed.index:
            n_genes = summary_indexed.loc[quantile, "n_genes"]
            if pd.notna(n_genes):
                group_labels.append(
                    f"{quantile}\n"
                    f"(n={int(n_genes)})"
                )
                continue
        group_labels.append(quantile)

    ax.set_xticks(positions)
    ax.set_xticklabels(group_labels)
    ax.set_xlabel("Gene-attention quantile")
    ax.set_ylabel("Mean log-normalized RNA expression in B cells")
    ax.set_title("B-cell RNA expression across gene-attention quantiles")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    y_min = float(np.nanmin(data[RNA_EXPRESSION_COLUMN]))
    y_max = float(np.nanmax(data[RNA_EXPRESSION_COLUMN]))
    y_range = max(y_max - y_min, 1e-6)
    bar_height = 0.03 * y_range
    bar_step = 0.08 * y_range
    current_y = y_max + 0.06 * y_range

    position_map = {
        quantile: position
        for quantile, position in zip(quantile_order, positions)
    }

    for row in pairwise_stats.itertuples(index=False):
        star_text = row.significance
        if star_text == "ns":
            continue
        x1 = position_map[row.group1]
        x2 = position_map[row.group2]
        add_significance_annotation(
            ax=ax,
            x1=x1,
            x2=x2,
            y=current_y,
            h=bar_height,
            text=star_text,
        )
        current_y += bar_step

    ax.set_ylim(y_min - 0.05 * y_range, current_y + 0.10 * y_range)

    if len(expression_arrays) >= 2:
        try:
            kw_stat, kw_p = stats.kruskal(*[arr for arr in expression_arrays if len(arr) > 0])
            ax.text(
                0.02,
                0.98,
                f"Kruskal-Wallis p = {kw_p:.2e}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=9,
            )
        except Exception:
            pass

    plt.tight_layout()

    pdf_file = output_prefix.with_suffix(".pdf")
    png_file = output_prefix.with_suffix(".png")

    plt.savefig(pdf_file, bbox_inches="tight")
    plt.savefig(png_file, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {pdf_file}")
    print(f"Saved: {png_file}")


# ============================================================
# 22. Run one profile analysis
# ============================================================

def run_profile_analysis(
    genes: pd.DataFrame,
    all_peaks: pd.DataFrame,
    group_column: str,
    all_matrix_output: Path,
    all_summary_output: Path,
) -> pd.DataFrame:

    all_profile = calculate_all_profiles(
        genes=genes,
        peaks=all_peaks,
        peak_set_name="All peaks",
        group_column=group_column,
    )

    all_profile.to_csv(
        all_matrix_output,
        index=False,
    )

    all_summary = summarize_profiles(
        profile_df=all_profile,
        group_column=group_column,
    )

    all_summary.to_csv(
        all_summary_output,
        index=False,
    )

    all_genes_with_peaks = (
        all_profile
        .groupby("gene_id")[
            "overlapping_peak_count"
        ]
        .sum()
        .gt(0)
        .sum()
    )

    print(
        f"[{group_column}] Genes with "
        f"at least one all-peak overlap: "
        f"{all_genes_with_peaks:,}/"
        f"{len(genes):,}"
    )

    return all_summary


# ============================================================
# 23. Main program
# ============================================================

def main() -> None:

    print("=" * 80)
    print(
        "B-cell ATAC accessibility metagene analysis"
    )
    print("=" * 80)

    print("\nInput files:")

    for path in [
        GENE_RANK_FILE,
        GTF_FILE,
        ATAC_H5AD_FILE,
        RNA_H5AD_FILE,
    ]:
        require_file(path)
        print(f"  {path}")

    standard_chromosomes = (
        get_standard_chromosomes()
    )

    # ========================================================
    # 1. Gene rank
    # ========================================================

    print("\n[1] Loading gene rank")

    gene_rank = load_gene_rank(
        GENE_RANK_FILE
    )

    print(
        f"Gene rank records: "
        f"{len(gene_rank):,}"
    )

    print(
        gene_rank.head(10).to_string(
            index=False
        )
    )

    # ========================================================
    # 2. GTF
    # ========================================================

    print("\n[2] Loading NCBI GTF")

    gene_coordinates = (
        load_gene_coordinates_from_gtf(
            GTF_FILE
        )
    )

    print(
        f"GTF unique gene symbols: "
        f"{len(gene_coordinates):,}"
    )

    # ========================================================
    # 3. Gene coordinate matching
    # ========================================================

    print(
        "\n[3] Matching ranked genes "
        "to GTF by gene symbol"
    )

    genes = gene_rank.merge(
        gene_coordinates,
        on="gene_symbol_upper",
        how="left",
        validate="many_to_one",
    )

    unmatched = (
        genes["chrom"].isna()
    )

    if unmatched.any():
        genes.loc[
            unmatched,
            [
                "gene_id",
                "gene_symbol",
                "gene_attention",
            ],
        ].to_csv(
            UNMATCHED_GENE_OUTPUT,
            index=False,
        )

        print(
            f"Warning: unmatched genes: "
            f"{unmatched.sum():,}/"
            f"{len(genes):,}"
        )

        print(
            genes.loc[
                unmatched,
                [
                    "gene_id",
                    "gene_symbol",
                ],
            ]
            .head(30)
            .to_string(index=False)
        )

        print(
            f"Saved: {UNMATCHED_GENE_OUTPUT}"
        )

    genes = genes.loc[
        ~unmatched
    ].copy()

    if KEEP_STANDARD_CHROMOSOMES:
        genes = genes[
            genes["chrom"].isin(
                standard_chromosomes
            )
        ].copy()

    if EXCLUDE_MITOCHONDRIAL:
        genes = genes[
            genes["chrom"] != "chrM"
        ].copy()

    if genes.empty:
        raise ValueError(
            "No genes remain after GTF matching."
        )

    genes = assign_quantiles(
        genes=genes,
        value_column="gene_attention",
        output_column=(
            "gene_attention_quantile"
        ),
    )

    genes = (
        genes.sort_values(
            "gene_attention",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    genes.to_csv(
        GENE_COORDINATE_OUTPUT,
        index=False,
    )

    print(
        f"Final genes retained: "
        f"{len(genes):,}"
    )

    print(
        "\nGene-attention quantiles:"
    )

    print(
        genes[
            "gene_attention_quantile"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )

    # ========================================================
    # 4. ATAC peak accessibility
    # ========================================================

    print("\n[4] Loading ATAC h5ad")

    adata_atac = ad.read_h5ad(
        ATAC_H5AD_FILE
    )

    print(adata_atac)

    print(
        "\nATAC var columns:"
    )

    print(
        adata_atac.var.columns.tolist()
    )

    print(
        "\n[5] Calculating B-cell "
        "peak accessibility"
    )

    all_peaks = (
        calculate_bcell_peak_accessibility(
            adata_atac
        )
    )

    if KEEP_STANDARD_CHROMOSOMES:
        all_peaks = all_peaks[
            all_peaks["chrom"].isin(
                standard_chromosomes
            )
        ].copy()

    if EXCLUDE_MITOCHONDRIAL:
        all_peaks = all_peaks[
            all_peaks["chrom"] != "chrM"
        ].copy()

    all_peaks = (
        all_peaks.sort_values(
            "peak_accessibility",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    if all_peaks.empty:
        raise ValueError(
            "No valid ATAC peaks remain."
        )

    all_peaks.to_csv(
        ALL_PEAK_ACTIVITY_OUTPUT,
        index=False,
    )

    print(
        f"Valid all peaks: "
        f"{len(all_peaks):,}"
    )

    print(
        f"Saved: {ALL_PEAK_ACTIVITY_OUTPUT}"
    )

    del adata_atac
    gc.collect()

    # ========================================================
    # 5. RNA expression
    # ========================================================

    print("\n[6] Loading RNA h5ad")

    adata_rna = ad.read_h5ad(
        RNA_H5AD_FILE
    )

    print(adata_rna)

    print(
        "\nRNA var columns:"
    )

    print(
        adata_rna.var.columns.tolist()
    )

    print(
        "\nRNA var_names examples:"
    )

    print(
        adata_rna.var_names[
            :10
        ].tolist()
    )

    print(
        "\n[7] Calculating RNA expression "
        "for ranked genes"
    )

    rna_expression = (
        calculate_bcell_gene_expression(
            adata_rna=adata_rna,
            genes=genes,
        )
    )

    rna_expression.to_csv(
        RNA_EXPRESSION_OUTPUT,
        index=False,
    )

    print(
        f"Saved: {RNA_EXPRESSION_OUTPUT}"
    )

    print(
        "\nRNA matching status:"
    )

    print(
        rna_expression[
            "rna_match_status"
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )

    genes_rna = genes.merge(
        rna_expression,
        on=[
            "gene_id",
            "gene_symbol",
        ],
        how="left",
        validate="one_to_one",
    )

    genes_rna = assign_quantiles(
        genes=genes_rna,
        value_column=(
            RNA_EXPRESSION_COLUMN
        ),
        output_column=(
            "rna_expression_quantile"
        ),
    )

    genes_rna = (
        genes_rna.sort_values(
            RNA_EXPRESSION_COLUMN,
            ascending=False,
        )
        .reset_index(drop=True)
    )

    genes_rna.to_csv(
        RNA_GENE_QUANTILE_OUTPUT,
        index=False,
    )

    print(
        f"Genes retained for RNA grouping: "
        f"{len(genes_rna):,}"
    )

    print(
        "\nRNA-expression quantiles:"
    )

    print(
        genes_rna[
            "rna_expression_quantile"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print(
        "\nTop RNA-expressed ranked genes:"
    )

    print(
        genes_rna[
            [
                "gene_id",
                "gene_symbol",
                RNA_EXPRESSION_COLUMN,
                "rna_fraction_expressing",
                "rna_expression_quantile",
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

    del adata_rna
    gc.collect()

    # ========================================================
    # 5.1 Gene-attention Q1-Q4 -> RNA expression
    # ========================================================

    print(
        "\n[8.1] RNA expression grouped by "
        "gene-attention Q1-Q4 (pairwise tests vs Q4)"
    )

    attention_expression_genes = (
        genes_rna[
            [
                "gene_id",
                "gene_symbol",
                "gene_attention",
                "gene_attention_quantile",
                RNA_EXPRESSION_COLUMN,
                "rna_fraction_expressing",
                "rna_pseudobulk_count",
                "rna_match_status",
                "rna_match_method",
                "rna_n_features",
            ]
        ]
        .dropna(
            subset=[
                "gene_attention_quantile",
                RNA_EXPRESSION_COLUMN,
            ]
        )
        .sort_values(
            [
                "gene_attention_quantile",
                "gene_attention",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    attention_expression_genes.to_csv(
        ATTENTION_EXPRESSION_GENE_OUTPUT,
        index=False,
    )

    attention_expression_summary = (
        summarize_expression_by_attention_quantile(
            genes_rna
        )
    )

    attention_expression_summary.to_csv(
        ATTENTION_EXPRESSION_SUMMARY_OUTPUT,
        index=False,
    )

    print(
        "\nB-cell RNA expression by "
        "gene-attention quantile:"
    )

    print(
        attention_expression_summary[
            [
                "gene_attention_quantile",
                "n_genes",
                "mean_expression",
                "median_expression",
                "standard_error",
                "expression_q25",
                "expression_q75",
            ]
        ]
        .to_string(index=False)
    )

    print(
        f"Saved: "
        f"{ATTENTION_EXPRESSION_GENE_OUTPUT}"
    )

    print(
        f"Saved: "
        f"{ATTENTION_EXPRESSION_SUMMARY_OUTPUT}"
    )

    plot_expression_by_attention_quantile(
        genes_rna=genes_rna,
        summary=attention_expression_summary,
        output_prefix=(
            ATTENTION_EXPRESSION_FIGURE_PREFIX
        ),
        stats_output=(
            ATTENTION_EXPRESSION_STATS_OUTPUT
        ),
    )

    # ========================================================
    # 6. Gene-attention grouping profiles
    # ========================================================

    print(
        "\n[8] Gene-attention Q1-Q4 profiles"
    )

    attention_all_summary = run_profile_analysis(
        genes=genes,
        all_peaks=all_peaks,
        group_column=(
            "gene_attention_quantile"
        ),
        all_matrix_output=(
            ATTENTION_ALL_MATRIX_OUTPUT
        ),
        all_summary_output=(
            ATTENTION_ALL_SUMMARY_OUTPUT
        ),
    )

    save_single_profile(
        summary=attention_all_summary,
        group_column=(
            "gene_attention_quantile"
        ),
        title=(
            "B-cell accessibility: all peaks, "
            "grouped by gene attention"
        ),
        legend_title=(
            "Gene attention\n"
            "Q1 low → Q4 high"
        ),
        output_prefix=(
            ATTENTION_ALL_FIGURE_PREFIX
        ),
    )

    # ========================================================
    # 7. RNA-expression grouping profiles
    # ========================================================

    print(
        "\n[9] RNA-expression Q1-Q4 profiles"
    )

    rna_all_summary = run_profile_analysis(
        genes=genes_rna,
        all_peaks=all_peaks,
        group_column=(
            "rna_expression_quantile"
        ),
        all_matrix_output=(
            RNA_ALL_MATRIX_OUTPUT
        ),
        all_summary_output=(
            RNA_ALL_SUMMARY_OUTPUT
        ),
    )

    save_single_profile(
        summary=rna_all_summary,
        group_column=(
            "rna_expression_quantile"
        ),
        title=(
            "B-cell accessibility: all peaks, "
            "grouped by RNA expression"
        ),
        legend_title=(
            "B-cell RNA expression\n"
            "Q1 low → Q4 high"
        ),
        output_prefix=(
            RNA_ALL_FIGURE_PREFIX
        ),
    )

    print("\n" + "=" * 80)
    print("Done")
    print("=" * 80)

    print(
        f"ATAC accessibility metric: "
        f"{ACCESSIBILITY_METRIC}"
    )

    print(
        f"Gene-attention genes: "
        f"{len(genes):,}"
    )

    print(
        f"RNA-expression genes: "
        f"{len(genes_rna):,}"
    )

    print(
        f"All peaks: {len(all_peaks):,}"
    )

    print(
        "Additional expression figure: "
        f"{ATTENTION_EXPRESSION_FIGURE_PREFIX.with_suffix('.pdf')}"
    )

    print(
        f"\nOutput directory:\n"
        f"{OUTPUT_DIR}"
    )


if __name__ == "__main__":
    main()